# Build Daily Relative Returns

## Objective

Build the validated daily analytical foundation for the **Indian Equity Weakness Monitor**.

For each sampled stock and trading date, calculate stock return, Nifty 50 return, equal-weighted leave-one-out sector return, active return versus Nifty, active return versus sector, and sector peer count.

## Sector benchmark

The sector benchmark is the equal-weighted daily return of all other available constituents in the frozen Nifty 500 sector peer universe. The focal stock is excluded. At least two valid peers are required.

## Limitations

Current/frozen Nifty 500 membership is used historically, so version 1 retains survivorship and membership bias. The Nifty 50 series uses Yahoo Finance with missing stock-calendar dates patched from the official NSE daily index archive.


In [ ]:
!pip -q install --upgrade yfinance nse-archives

import time, json
from io import BytesIO
import numpy as np
import pandas as pd
import requests
import yfinance as yf
from nsedata import nse

START_DATE = '2018-01-01'
RUN_DATE = pd.Timestamp.today().normalize()
END_DATE = (RUN_DATE + pd.Timedelta(days=1)).strftime('%Y-%m-%d')
NIFTY_SYMBOL = '^NSEI'
BATCH_SIZE = 40
MAX_RETRIES = 3
MIN_VALID_SECTOR_PEERS = 2

SAMPLED_UNIVERSE_URL = 'https://raw.githubusercontent.com/chinmay227/indian-market-internals/main/data/reference/sampled_nifty500_universe_100.csv'
SECTOR_SNAPSHOT_URL = 'https://raw.githubusercontent.com/chinmay227/indian-market-internals/main/data/reference/nifty500_sector_universe_snapshot.csv'
NIFTY500_URL = 'https://www.niftyindices.com/IndexConstituent/ind_nifty500list.csv'

sampled_universe = pd.read_csv(SAMPLED_UNIVERSE_URL)
assert len(sampled_universe) == 100 and sampled_universe['ticker'].nunique() == 100

try:
    r = requests.get(SECTOR_SNAPSHOT_URL, headers={'User-Agent':'Mozilla/5.0'}, timeout=15)
    r.raise_for_status()
    full_universe = pd.read_csv(BytesIO(r.content))
    sector_universe_source = 'frozen GitHub snapshot'
except Exception:
    r = requests.get(NIFTY500_URL, headers={'User-Agent':'Mozilla/5.0'}, timeout=30)
    r.raise_for_status()
    raw = pd.read_csv(BytesIO(r.content))
    full_universe = raw[['Company Name','Symbol','Industry','ISIN Code']].rename(columns={'Company Name':'company_name','Symbol':'ticker','Industry':'sector','ISIN Code':'isin'}).copy()
    sector_universe_source = 'live official Nifty 500 file'

assert len(full_universe) == 500 and full_universe['ticker'].nunique() == 500
missing_sample = sorted(set(sampled_universe['ticker']) - set(full_universe['ticker']))
if missing_sample:
    raise ValueError(f'Sampled stocks missing from sector universe: {missing_sample}')

full_universe['yf_ticker'] = full_universe['ticker'] + '.NS'
yf_to_ticker = dict(zip(full_universe['yf_ticker'], full_universe['ticker']))
print('Sample:', len(sampled_universe), 'Sector universe:', len(full_universe), 'Source:', sector_universe_source)


In [ ]:
def extract_close_frame(raw, expected_symbols):
    if raw is None or raw.empty:
        return pd.DataFrame()
    if isinstance(raw.columns, pd.MultiIndex):
        l0 = raw.columns.get_level_values(0)
        l1 = raw.columns.get_level_values(1)
        if 'Close' in l0:
            close = raw['Close'].copy()
        elif 'Close' in l1:
            close = raw.xs('Close', axis=1, level=1).copy()
        else:
            raise ValueError('Close field not found')
    else:
        close = raw[['Close']].copy()
        if len(expected_symbols) == 1:
            close.columns = expected_symbols
    if isinstance(close, pd.Series):
        close = close.to_frame()
    close.index = pd.to_datetime(close.index)
    return close.sort_index()

def download_batch(symbols):
    last_error = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            raw = yf.download(symbols, start=START_DATE, end=END_DATE, auto_adjust=True, progress=False, threads=True, group_by='column')
            close = extract_close_frame(raw, symbols)
            if close.empty:
                raise ValueError('empty Yahoo response')
            return close
        except Exception as e:
            last_error = e
            if attempt < MAX_RETRIES:
                time.sleep(2 * attempt)
    raise RuntimeError(last_error)

symbols = full_universe['yf_ticker'].tolist()
parts = []
for start in range(0, len(symbols), BATCH_SIZE):
    batch = symbols[start:start+BATCH_SIZE]
    parts.append(download_batch(batch))
    time.sleep(0.8)

stock_close_yahoo = pd.concat(parts, axis=1)
stock_close_yahoo = stock_close_yahoo.loc[:, ~stock_close_yahoo.columns.duplicated()].sort_index()
stock_close = stock_close_yahoo.rename(columns=yf_to_ticker)
stock_close = stock_close[[t for t in full_universe['ticker'] if t in stock_close.columns]].copy()

coverage_rows = []
for ticker in full_universe['ticker']:
    if ticker not in stock_close.columns:
        coverage_rows.append({'ticker':ticker,'observations':0,'first_price_date':pd.NaT,'last_price_date':pd.NaT})
    else:
        s = stock_close[ticker]
        coverage_rows.append({'ticker':ticker,'observations':int(s.notna().sum()),'first_price_date':s.first_valid_index(),'last_price_date':s.last_valid_index()})
stock_price_coverage = pd.DataFrame(coverage_rows)

nifty_raw = yf.download(NIFTY_SYMBOL, start=START_DATE, end=END_DATE, auto_adjust=True, progress=False, threads=False)
nifty_frame = extract_close_frame(nifty_raw, [NIFTY_SYMBOL])
nifty_close_yahoo = (nifty_frame[NIFTY_SYMBOL] if NIFTY_SYMBOL in nifty_frame.columns else nifty_frame.iloc[:,0]).copy()

sampled_downloaded = [t for t in sampled_universe['ticker'] if t in stock_close.columns]
sample_stock_calendar = stock_close[sampled_downloaded].dropna(how='all').index
missing_dates = sample_stock_calendar.difference(nifty_close_yahoo.dropna().index).sort_values()
patched = {}
failed_patch_dates = []
for date in missing_dates:
    ds = pd.Timestamp(date).strftime('%Y-%m-%d')
    try:
        archive = nse.get('capital_market','indices','ind_close_all',ds)
        names = archive['Index Name'].astype(str).str.strip().str.upper()
        mask = names.eq('NIFTY 50')
        raw_close = archive.loc[mask,'Closing Index Value'].iloc[0]
        val = pd.to_numeric(str(raw_close).replace(',',''), errors='coerce')
        if pd.notna(val): patched[pd.Timestamp(date)] = float(val)
        else: failed_patch_dates.append(ds)
    except Exception:
        failed_patch_dates.append(ds)
    time.sleep(0.15)

nifty_close = nifty_close_yahoo.copy()
for date, value in patched.items():
    nifty_close.loc[date] = value
nifty_close = nifty_close.sort_index()

stock_returns = stock_close.pct_change(fill_method=None)
nifty_return = nifty_close.pct_change(fill_method=None)
nifty_return.name = 'nifty_return'
print('Nifty closes patched:', len(patched), 'Patch failures:', len(failed_patch_dates))


In [ ]:
sector_stats = {}
for sector, members in full_universe.groupby('sector'):
    tickers = [t for t in members['ticker'] if t in stock_returns.columns]
    matrix = stock_returns[tickers]
    sector_stats[sector] = {'sum':matrix.sum(axis=1, min_count=1), 'count':matrix.notna().sum(axis=1)}

master_index = stock_returns.index.union(nifty_return.index).sort_values()
parts = []
for ticker, meta in sampled_universe.set_index('ticker').iterrows():
    sector = meta['sector']
    stock_r = stock_returns[ticker].reindex(master_index) if ticker in stock_returns.columns else pd.Series(np.nan, index=master_index)
    nifty_r = nifty_return.reindex(master_index)
    sector_sum = sector_stats[sector]['sum'].reindex(master_index)
    sector_count = sector_stats[sector]['count'].reindex(master_index, fill_value=0)
    focal_available = stock_r.notna().astype(int)
    peer_count = (sector_count - focal_available).clip(lower=0)
    peer_sum = sector_sum.fillna(0) - stock_r.fillna(0)
    peer_return = peer_sum / peer_count.replace(0, np.nan)
    peer_return = peer_return.where(peer_count >= MIN_VALID_SECTOR_PEERS)
    frame = pd.DataFrame({'date':master_index,'ticker':ticker,'company_name':meta['company_name'],'sector':sector,'market_cap_rank':meta['market_cap_rank'],'market_cap_stratum':meta['market_cap_stratum'],'stock_return':stock_r.to_numpy(),'nifty_return':nifty_r.to_numpy(),'sector_peer_return':peer_return.to_numpy(),'sector_peer_count':peer_count.to_numpy()})
    frame['active_return_vs_nifty'] = frame['stock_return'] - frame['nifty_return']
    frame['active_return_vs_sector'] = frame['stock_return'] - frame['sector_peer_return']
    parts.append(frame[frame['stock_return'].notna()].copy())

daily_relative_returns = pd.concat(parts, ignore_index=True).sort_values(['date','ticker']).reset_index(drop=True)
duplicates = daily_relative_returns.duplicated(['date','ticker']).sum()
assert duplicates == 0

market_available = daily_relative_returns['active_return_vs_nifty'].notna()
sector_available = daily_relative_returns['active_return_vs_sector'].notna()

print('='*80)
print('DAILY ANALYTICAL DATASET VALIDATION')
print('='*80)
print('Rows:', len(daily_relative_returns))
print('Unique sampled stocks represented:', daily_relative_returns['ticker'].nunique())
print('First date:', daily_relative_returns['date'].min())
print('Last date:', daily_relative_returns['date'].max())
print('Duplicate stock-date rows:', duplicates)
print('Market-relative availability:', round(market_available.mean()*100,4), '%')
print('Market-relative rows missing:', int((~market_available).sum()))
print('Sector-relative availability:', round(sector_available.mean()*100,4), '%')
print('Sector-relative rows missing:', int((~sector_available).sum()))

peer_summary = daily_relative_returns.groupby('sector')['sector_peer_count'].agg(minimum='min', median='median', maximum='max').sort_values('minimum')
display(peer_summary)

# Independent spot checks
check_rows = daily_relative_returns[daily_relative_returns['sector_peer_return'].notna()].sort_values('date').tail(300).drop_duplicates('ticker').tail(5)
checks = []
for row in check_rows.itertuples(index=False):
    peers = full_universe.loc[full_universe['sector'].eq(row.sector) & ~full_universe['ticker'].eq(row.ticker),'ticker'].tolist()
    available = [p for p in peers if p in stock_returns.columns]
    direct = stock_returns.reindex(index=[pd.Timestamp(row.date)], columns=available).iloc[0].dropna()
    direct_mean = direct.mean() if len(direct) >= MIN_VALID_SECTOR_PEERS else np.nan
    diff = abs(direct_mean-row.sector_peer_return) if pd.notna(direct_mean) else np.nan
    checks.append({'date':row.date,'ticker':row.ticker,'sector':row.sector,'direct_peer_count':len(direct),'stored_peer_count':int(row.sector_peer_count),'absolute_difference':diff})
spot_checks = pd.DataFrame(checks)
display(spot_checks)
valid = spot_checks['absolute_difference'].dropna()
if len(valid): assert valid.max() < 1e-12
assert (spot_checks['direct_peer_count'] == spot_checks['stored_peer_count']).all()
print('Spot checks passed.')

daily_relative_returns.to_csv('daily_relative_returns_v1.csv', index=False)
stock_price_coverage.to_csv('nifty500_stock_price_coverage.csv', index=False)
full_universe.to_csv('nifty500_sector_universe_snapshot.csv', index=False)
metadata = {'run_date':str(RUN_DATE.date()),'sample_size':100,'sector_universe_size':500,'sector_universe_source':sector_universe_source,'nifty_closes_patched':len(patched),'nifty_patch_failures':failed_patch_dates,'min_valid_sector_peers':MIN_VALID_SECTOR_PEERS}
with open('relative_returns_run_metadata.json','w') as f: json.dump(metadata,f,indent=2)
print('Saved daily_relative_returns_v1.csv, nifty500_stock_price_coverage.csv, nifty500_sector_universe_snapshot.csv, relative_returns_run_metadata.json')
